In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from statsforecast.models import AutoARIMA
from statsforecast import StatsForecast

d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("D:\\Users\\Md Mahfuzur Rahman\\Desktop\\Research\\for_me\\cleaned_combined_data.csv", parse_dates=["datetime"])
df["unique_id"] = "BD"
df.head()

,datetime,demand_mw,load_shedding,temp_mean,wspd_mean,unique_id
0,2020-01-01 03:00:00,5351,0,18.65,1.80,BD
1,2020-01-01 06:00:00,5346,0,23.73,3.00,BD
2,2020-01-01 09:00:00,6654,0,25.25,4.58,BD
3,2020-01-01 12:00:00,6750,0,21.05,2.17,BD
4,2020-01-01 15:00:00,6546,0,18.82,0.30,BD


In [4]:
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['dayofyear'] = df['datetime'].dt.dayofyear

# -----------------------------
# Cyclic time features
# -----------------------------

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)


# -----------------------------
# Lag features
# -----------------------------

for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]:
    df[f'lag{lag}'] = df['demand_mw'].shift(lag)


# -----------------------------
# Rolling features
# IMPORTANT: shift(1)
# prevents target leakage
# -----------------------------

past_demand = df['demand_mw'].shift(1)

for window in [6, 12, 24, 48, 168]:

    df[f'roll{window}_mean'] = (
        past_demand.rolling(window).mean()
    )

    df[f'roll{window}_std'] = (
        past_demand.rolling(window).std()
    )


# Remove NaN
df = df.dropna().reset_index(drop=True)

In [5]:
df_ext = (
    df[["datetime", "demand_mw",

        'hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','doy_sin','doy_cos',
    
        'load_shedding','temp_mean','wspd_mean',
    
        'lag1','lag2','lag3','lag6','lag12','lag24','lag48','lag72','lag168','lag336',
    
        'roll6_mean','roll6_std','roll12_mean','roll12_std','roll24_mean','roll24_std',
        'roll48_mean','roll48_std','roll168_mean','roll168_std']]
    .rename(columns={"datetime": "ds", "demand_mw": "y"})
    .assign(unique_id=df["unique_id"])
)

In [11]:
def metrics(name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    acc  = max(0.0, 1.0 - mape / 100.0) * 100
    return dict(Model=name,
                Accuracy=round(acc, 3),
                MAE=round(mae, 1),
                RMSE=round(rmse, 1),
                MAPE=round(mape, 3),
                R2=round(r2, 4))

results = []  # Store results for all models

In [8]:
# Sort chronologically
sorted_df = df_ext.sort_values("ds").reset_index(drop=True)

# Last 168 rows for testing
test_size = 168

test = sorted_df.iloc[-test_size:].copy()

# Remaining data
remaining_df = sorted_df.iloc[:-test_size].copy()

# Last 20% of the remaining data for training
train_size = int(len(remaining_df) * 0.20)

train = remaining_df.iloc[-train_size:].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("Train time range:",
      train["ds"].min(), "to", train["ds"].max())

print("Test time range:",
      test["ds"].min(), "to", test["ds"].max())

print("\nTrain tail:")
print(train.tail())

print("\nTest:")
print(test)

Train shape: (8421, 34)
Test shape: (168, 34)
Train time range: 2024-12-18 08:00:00 to 2025-12-24 16:00:00
Test time range: 2025-12-24 17:00:00 to 2025-12-31 23:00:00

Train tail:
                       ds      y      hour_sin  hour_cos   dow_sin   dow_cos  \
42102 2025-12-24 12:00:00  10134  1.224647e-16 -1.000000  0.974928 -0.222521   
42103 2025-12-24 13:00:00  10545 -2.588190e-01 -0.965926  0.974928 -0.222521   
42104 2025-12-24 14:00:00  10209 -5.000000e-01 -0.866025  0.974928 -0.222521   
42105 2025-12-24 15:00:00   9967 -7.071068e-01 -0.707107  0.974928 -0.222521   
42106 2025-12-24 16:00:00   9629 -8.660254e-01 -0.500000  0.974928 -0.222521   

          month_sin  month_cos   doy_sin   doy_cos  ...   roll6_std  \
42102 -2.449294e-16        1.0 -0.120208  0.992749  ...  748.203560   
42103 -2.449294e-16        1.0 -0.120208  0.992749  ...  649.226514   
42104 -2.449294e-16        1.0 -0.120208  0.992749  ...  515.790171   
42105 -2.449294e-16        1.0 -0.120208  0.992749  ...

In [9]:
futr_exog_df = test.drop(["y"], axis=1)
futr_exog_df.head()
futr_exog_df.shape

(168, 33)

In [12]:

# ============================================================
# 1. Different AutoARIMA configurations to test
# ============================================================

param_grid = [
    # Small search space - faster
    {
        "season_length": 24,
        "max_p": 2,
        "max_q": 2,
        "max_P": 1,
        "max_Q": 1,
        "max_order": 4,
        "approximation": True
    },

    # Medium search space
    {
        "season_length": 24,
        "max_p": 3,
        "max_q": 3,
        "max_P": 1,
        "max_Q": 1,
        "max_order": 5,
        "approximation": True
    },

    {
        "season_length": 24,
        "max_p": 4,
        "max_q": 4,
        "max_P": 2,
        "max_Q": 2,
        "max_order": 6,
        "approximation": True
    },

    # Larger search space
    {
        "season_length": 24,
        "max_p": 5,
        "max_q": 5,
        "max_P": 2,
        "max_Q": 2,
        "max_order": 8,
        "approximation": False
    }
]


# ============================================================
# 2. Time-series cross-validation
# ============================================================

results = []

for i, params in enumerate(param_grid):

    print(f"\nRunning configuration {i+1}/{len(param_grid)}")
    print(params)

    # Create AutoARIMA using current parameters
    model = AutoARIMA(
        season_length=params["season_length"],
        max_p=params["max_p"],
        max_q=params["max_q"],
        max_P=params["max_P"],
        max_Q=params["max_Q"],
        max_order=params["max_order"],
        approximation=params["approximation"],
        alias="SARIMA"
    )

    # Create StatsForecast object
    sf = StatsForecast(
        models=[model],
        freq="H"
    )

    # --------------------------------------------------------
    # Cross-validation
    #
    # h=24      -> predict next 24 hours
    # n_windows=5 -> use 5 historical validation windows
    # step_size=24 -> move forward 24 hours each time
    # --------------------------------------------------------

    cv_df = sf.cross_validation(
        df=train,
        h=24,
        n_windows=5,
        step_size=24,
        refit=True
    )

    # --------------------------------------------------------
    # Calculate RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(
            cv_df["y"],
            cv_df["SARIMA"]
        )
    )

    results.append({
        **params,
        "RMSE": rmse
    })

    print(f"RMSE = {rmse:.4f}")


# ============================================================
# 3. Show the best configuration
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

print("\n========== BEST CONFIGURATION ==========")
print(results_df.iloc[0])

print("\n========== ALL RESULTS ==========")
print(results_df)


Running configuration 1/4
{'season_length': 24, 'max_p': 2, 'max_q': 2, 'max_P': 1, 'max_Q': 1, 'max_order': 4, 'approximation': True}


d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 1]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: p

RMSE = 393.1144

Running configuration 2/4
{'season_length': 24, 'max_p': 3, 'max_q': 3, 'max_P': 1, 'max_Q': 1, 'max_order': 5, 'approximation': True}


d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 1]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: p

RMSE = 398.9561

Running configuration 3/4
{'season_length': 24, 'max_p': 4, 'max_q': 4, 'max_P': 2, 'max_Q': 2, 'max_order': 6, 'approximation': True}


d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 1]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: possible convergence problem: minimize gave code 2]
  warnings.warn(
d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\statsforecast\arima.py:672: UserWarning: p

RMSE = 403.7564

Running configuration 4/4
{'season_length': 24, 'max_p': 5, 'max_q': 5, 'max_P': 2, 'max_Q': 2, 'max_order': 8, 'approximation': False}


KeyboardInterrupt: 

###
Running configuration 1/4 - RMSE = 393.1144

{'season_length': 24, 'max_p': 2, 'max_q': 2, 'max_P': 1, 'max_Q': 1, 'max_order': 4, 'approximation': True}


Running configuration 2/4 - RMSE = 398.9561

{'season_length': 24, 'max_p': 3, 'max_q': 3, 'max_P': 1, 'max_Q': 1, 'max_order': 5, 'approximation': True}


Running configuration 3/4 - RMSE = 403.7564

{'season_length': 24, 'max_p': 4, 'max_q': 4, 'max_P': 2, 'max_Q': 2, 'max_order': 6, 'approximation': True}

###

In [13]:
# Get best parameters
best = results_df.iloc[0]

print("Best parameters:")
print(best)


# ------------------------------------------------------------
# Train final model using the best configuration
# ------------------------------------------------------------

final_model = AutoARIMA(
    season_length=int(best["season_length"]),
    max_p=int(best["max_p"]),
    max_q=int(best["max_q"]),
    max_P=int(best["max_P"]),
    max_Q=int(best["max_Q"]),
    max_order=int(best["max_order"]),
    approximation=bool(best["approximation"]),
    alias="SARIMA_exog"
)

sf_final = StatsForecast(
    models=[final_model],
    freq="H"
)

sf_final.fit(df=train)

NameError: name 'results_df' is not defined

In [14]:
from sklearn.ensemble import GradientBoostingRegressor

In [15]:
print("Loading data...")
df = pd.read_csv('D:\\Users\\Md Mahfuzur Rahman\\Desktop\\Research\\for_me\\cleaned_combined_data.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

Loading data...


In [16]:
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['dayofyear'] = df['datetime'].dt.dayofyear

# -----------------------------
# Cyclic time features
# -----------------------------

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)


# -----------------------------
# Lag features
# -----------------------------

for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]:
    df[f'lag{lag}'] = df['demand_mw'].shift(lag)


# -----------------------------
# Rolling features
# IMPORTANT: shift(1)
# prevents target leakage
# -----------------------------

past_demand = df['demand_mw'].shift(1)

for window in [6, 12, 24, 48, 168]:

    df[f'roll{window}_mean'] = (
        past_demand.rolling(window).mean()
    )

    df[f'roll{window}_std'] = (
        past_demand.rolling(window).std()
    )


# Remove NaN
df = df.dropna().reset_index(drop=True)

In [17]:
df.shape

(42275, 37)

In [18]:
FEATURES = [
    'hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','doy_sin','doy_cos',

    'load_shedding','temp_mean','wspd_mean',

    'lag1','lag2','lag3','lag6','lag12','lag24','lag48','lag72','lag168','lag336',

    'roll6_mean','roll6_std','roll12_mean','roll12_std','roll24_mean','roll24_std',
    'roll48_mean','roll48_std','roll168_mean','roll168_std'
]

TARGET = 'demand_mw'

In [19]:
from sklearn.model_selection import train_test_split
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Reset indices for easier access
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Create aliases for backward compatibility
X_tr, X_te = X_train, X_test
y_tr, y_te = y_train, y_test

print(f"Train rows: {len(X_train):,}   Test rows: {len(X_test):,}")

Train rows: 33,820   Test rows: 8,455


In [20]:
from sklearn.preprocessing import StandardScaler
import torch

X_scaler = StandardScaler()

X_tr_sc = X_scaler.fit_transform(X_tr)
X_te_sc = X_scaler.transform(X_te)

# PyTorch tensors
X_tr = torch.tensor(X_tr_sc, dtype=torch.float32)
X_te = torch.tensor(X_te_sc, dtype=torch.float32)

y_tr = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1)
y_te = torch.tensor(y_te, dtype=torch.float32).reshape(-1, 1)

In [23]:
import optuna
import xgboost as xgb
import numpy as np

from sklearn.metrics import mean_squared_error


# ============================================================
# 1. Create a time-based validation split
# ============================================================
# IMPORTANT:
# We use only X_tr / y_tr for Optuna.
# X_te / y_te stays completely untouched.

val_size = int(len(X_tr) * 0.20)

# Works with both pandas DataFrame/Series and NumPy arrays
if hasattr(X_tr, "iloc"):
    X_train_cv = X_tr.iloc[:-val_size]
    X_val_cv   = X_tr.iloc[-val_size:]

    y_train_cv = y_tr.iloc[:-val_size]
    y_val_cv   = y_tr.iloc[-val_size:]
else:
    X_train_cv = X_tr[:-val_size]
    X_val_cv   = X_tr[-val_size:]

    y_train_cv = y_tr[:-val_size]
    y_val_cv   = y_tr[-val_size:]


# ============================================================
# 2. Optuna objective function
# ============================================================

def objective(trial):

    # --------------------------------------------------------
    # Optuna chooses these hyperparameters automatically
    # --------------------------------------------------------

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 300, 1200, step=100
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.10, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 8
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 15
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.70, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.70, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 1.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1.0, 10.0, log=True
        )
    }


    # --------------------------------------------------------
    # Create XGBoost model
    # --------------------------------------------------------

    model = xgb.XGBRegressor(
        objective="reg:squarederror",

        **params,

        random_state=42,
        tree_method="hist",
        n_jobs=-1
    )


    # --------------------------------------------------------
    # Train ONLY on earlier data
    # --------------------------------------------------------

    model.fit(
        X_train_cv,
        y_train_cv,
        verbose=False
    )


    # --------------------------------------------------------
    # Predict validation period
    # --------------------------------------------------------

    pred = model.predict(X_val_cv)


    # --------------------------------------------------------
    # Calculate RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(y_val_cv, pred)
    )

    return rmse


# ============================================================
# 3. Create Optuna study
# ============================================================

study = optuna.create_study(
    direction="minimize",
    study_name="XGBoost_Electricity_Demand"
)


# ============================================================
# 4. Start hyperparameter optimization
# ============================================================

study.optimize(
    objective,
    n_trials=50,       # Try 50 different combinations
    show_progress_bar=True
)


# ============================================================
# 5. Display best parameters
# ============================================================

print("\n==========================================")
print("BEST XGBOOST PARAMETERS")
print("==========================================")

print("Best RMSE:")
print(study.best_value)

print("\nBest Parameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

[I 2026-08-22 01:08:50,295] A new study created in memory with name: XGBoost_Electricity_Demand
Best trial: 0. Best value: 336.043:   2%|▏         | 1/50 [00:01<00:57,  1.17s/it]

[I 2026-08-22 01:08:51,467] Trial 0 finished with value: 336.04301249988816 and parameters: {'n_estimators': 300, 'learning_rate': 0.09100358391708445, 'max_depth': 4, 'min_child_weight': 13, 'subsample': 0.8266917415465012, 'colsample_bytree': 0.835413985646015, 'gamma': 0.6126005677462565, 'reg_alpha': 1.4890087037719996e-05, 'reg_lambda': 2.4026501357231327}. Best is trial 0 with value: 336.04301249988816.


Best trial: 1. Best value: 335.989:   4%|▍         | 2/50 [00:04<02:08,  2.69s/it]

[I 2026-08-22 01:08:55,217] Trial 1 finished with value: 335.9890134922569 and parameters: {'n_estimators': 1200, 'learning_rate': 0.010771421158631457, 'max_depth': 5, 'min_child_weight': 8, 'subsample': 0.8481172195204313, 'colsample_bytree': 0.8077821039021381, 'gamma': 0.5067734583747873, 'reg_alpha': 1.8623306316887333e-06, 'reg_lambda': 4.166727524482641}. Best is trial 1 with value: 335.9890134922569.


Best trial: 1. Best value: 335.989:   6%|▌         | 3/50 [00:05<01:25,  1.82s/it]

...

[I 2026-08-22 01:11:18,179] Trial 47 finished with value: 335.18054836162554 and parameters: {'n_estimators': 900, 'learning_rate': 0.02729400050970018, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.7833858977861292, 'colsample_bytree': 0.9170427346352695, 'gamma': 0.05976623290073155, 'reg_alpha': 1.250378770094296e-08, 'reg_lambda': 3.14030410092545}. Best is trial 39 with value: 326.8935821678058.


Best trial: 39. Best value: 326.894:  98%|█████████▊| 49/50 [02:30<00:02,  2.93s/it]

[I 2026-08-22 01:11:20,924] Trial 48 finished with value: 337.5070138160095 and parameters: {'n_estimators': 700, 'learning_rate': 0.024757046521097677, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.7518054694444615, 'colsample_bytree': 0.9843712832079435, 'gamma': 0.09237322253375163, 'reg_alpha': 1.044224972426472e-08, 'reg_lambda': 3.813838508883023}. Best is trial 39 with value: 326.8935821678058.


Best trial: 39. Best value: 326.894: 100%|██████████| 50/50 [02:32<00:00,  3.05s/it]

[I 2026-08-22 01:11:22,713] Trial 49 finished with value: 335.06959398384686 and parameters: {'n_estimators': 600, 'learning_rate': 0.017510501000403963, 'max_depth': 5, 'min_child_weight': 11, 'subsample': 0.7362621955406915, 'colsample_bytree': 0.9287011594084164, 'gamma': 0.17569314564422944, 'reg_alpha': 2.5469309101377956e-08, 'reg_lambda': 2.8844297443747817}. Best is trial 39 with value: 326.8935821678058.

BEST XGBOOST PARAMETERS
Best RMSE:
326.8935821678058

Best Parameters:
n_estimators: 800
learning_rate: 0.026340059352236846
max_depth: 5
min_child_weight: 11
subsample: 0.7263709147024497
colsample_bytree: 0.9493553396095371
gamma: 0.03818554358601167
reg_alpha: 4.0561758823329404e-08
reg_lambda: 2.310596649947486


In [24]:
# ============================================================
# Gradient Boosting + Optuna Hyperparameter Tuning
# ============================================================

import optuna
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error


# ============================================================
# 1. Create time-based validation set
# ============================================================
# IMPORTANT:
# We only use X_tr_sc / y_tr for tuning.
# X_te_sc / y_te remains completely untouched.

val_size = int(len(X_tr_sc) * 0.20)

X_train_cv = X_tr_sc[:-val_size]
X_val_cv   = X_tr_sc[-val_size:]

y_train_cv = y_tr[:-val_size]
y_val_cv   = y_tr[-val_size:]


# ============================================================
# 2. Optuna objective function
# ============================================================

def objective(trial):

    # --------------------------------------------------------
    # Optuna will automatically search these parameters
    # --------------------------------------------------------

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            800,
            step=50
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.20,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            30
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            20
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "max_features": trial.suggest_categorical(
            "max_features",
            [None, "sqrt", "log2"]
        ),

        "loss": trial.suggest_categorical(
            "loss",
            ["squared_error", "huber"]
        )
    }


    # --------------------------------------------------------
    # Create Gradient Boosting model
    # --------------------------------------------------------

    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )


    # --------------------------------------------------------
    # Train on earlier observations
    # --------------------------------------------------------

    model.fit(
        X_train_cv,
        y_train_cv
    )


    # --------------------------------------------------------
    # Predict validation period
    # --------------------------------------------------------

    pred = model.predict(X_val_cv)


    # --------------------------------------------------------
    # RMSE = Optuna objective
    # Lower RMSE is better
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(
            y_val_cv,
            pred
        )
    )

    return rmse


# ============================================================
# 3. Create Optuna study
# ============================================================

study_gb = optuna.create_study(
    direction="minimize",
    study_name="GradientBoosting_Electricity_Demand"
)


# ============================================================
# 4. Run optimization
# ============================================================

study_gb.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)


# ============================================================
# 5. Show best result
# ============================================================

print("\n========================================")
print("BEST GRADIENT BOOSTING PARAMETERS")
print("========================================")

print("Best Validation RMSE:")
print(study_gb.best_value)

print("\nBest Parameters:")

for key, value in study_gb.best_params.items():
    print(f"{key}: {value}")

[I 2026-08-22 01:22:50,831] A new study created in memory with name: GradientBoosting_Electricity_Demand
  0%|          | 0/50 [00:00<?, ?it/s]d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\sklearn\ensemble\_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?
Best trial: 0. Best value: 755.259:   2%|▏         | 1/50 [00:08<07:16,  8.92s/it]d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\sklearn\ensemble\_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


[I 2026-08-22 01:22:59,749] Trial 0 finished with value: 755.2585383758011 and parameters: {'n_estimators': 100, 'learning_rate': 0.020002719641915644, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 5, 'subsample': 0.605662163491419, 'max_features': 'sqrt', 'loss': 'huber'}. Best is trial 0 with value: 755.2585383758011.


Best trial: 1. Best value: 331.525:   4%|▍         | 2/50 [02:03<56:46, 70.96s/it]d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\sklearn\ensemble\_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


[I 2026-08-22 01:24:54,139] Trial 1 finished with value: 331.5248896641915 and parameters: {'n_estimators': 700, 'learning_rate': 0.05680505060665195, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 20, 'subsample': 0.75476247111943, 'max_features': None, 'loss': 'squared_error'}. Best is trial 1 with value: 331.5248896641915.


Best trial: 2. Best value: 319.8:   6%|▌         | 3/50 [04:39<1:26:14, 110.09s/it]d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\sklearn\ensemble\_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


...

[I 2026-08-22 03:06:17,597] Trial 47 finished with value: 316.4284853723778 and parameters: {'n_estimators': 500, 'learning_rate': 0.08971386666433546, 'max_depth': 4, 'min_samples_split': 14, 'min_samples_leaf': 18, 'subsample': 0.6045218989221692, 'max_features': None, 'loss': 'huber'}. Best is trial 6 with value: 309.9218432883534.


Best trial: 6. Best value: 309.922:  98%|█████████▊| 49/50 [1:46:26<02:34, 154.69s/it]d:\Users\Md Mahfuzur Rahman\Desktop\Research\venv\Lib\site-packages\sklearn\ensemble\_gb.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


[I 2026-08-22 03:09:17,622] Trial 48 finished with value: 326.6077308596387 and parameters: {'n_estimators': 800, 'learning_rate': 0.03885954555145841, 'max_depth': 3, 'min_samples_split': 19, 'min_samples_leaf': 17, 'subsample': 0.6842522601080382, 'max_features': None, 'loss': 'huber'}. Best is trial 6 with value: 309.9218432883534.


Best trial: 6. Best value: 309.922: 100%|██████████| 50/50 [1:49:11<00:00, 131.03s/it]

[I 2026-08-22 03:12:02,491] Trial 49 finished with value: 313.5304970029526 and parameters: {'n_estimators': 550, 'learning_rate': 0.05261011922863688, 'max_depth': 4, 'min_samples_split': 23, 'min_samples_leaf': 19, 'subsample': 0.717044747025304, 'max_features': None, 'loss': 'huber'}. Best is trial 6 with value: 309.9218432883534.

BEST GRADIENT BOOSTING PARAMETERS
Best Validation RMSE:
309.9218432883534

Best Parameters:
n_estimators: 700
learning_rate: 0.05128257080305278
max_depth: 4
min_samples_split: 20
min_samples_leaf: 17
subsample: 0.6972258345651443
max_features: None
loss: huber


In [ ]:
# ============================================================
# 6. Train FINAL Gradient Boosting model
# ============================================================

best_gb_params = study_gb.best_params

m_gb = GradientBoostingRegressor(
    **best_gb_params,
    random_state=42
)


# Train on ALL training data
m_gb.fit(
    X_tr_sc,
    y_tr
)


# ============================================================
# 7. Predict on untouched test data
# ============================================================

pred_gb = m_gb.predict(X_te_sc)


# ============================================================
# 8. Evaluate using your existing metrics function
# ============================================================

results.append(
    metrics(
        "Gradient Boosting Optuna",
        y_te,
        pred_gb
    )
)


print("\nGradient Boosting testing completed.")